# Contar usuarios por género

Cuenta cuántos usuarios hay según el género de su nombre. Lo decide una IA que corre en tu ordenador.

## La petición

`GET /users/gender-count`, sin cuerpo.

## Qué pasa en la base

Solo lee. No cambia nada.

## Las respuestas

| Estado | Qué se ve |
| --- | --- |
| 200 | Los tres recuentos: `women`, `men` y `genderUndefined`. Si no hay usuarios, salen tres ceros |
| 500 | `Could not classify the users`. No se ha llegado a la base o a la IA |

## Qué tiene que estar en marcha

Antes de ejecutarlo, lanza `pnpm start` y `pnpm node-http`, cada uno en su terminal.

También necesita Ollama con el modelo `llama3.2`, salvo si no hay usuarios. La respuesta puede tardar un poco, y el recuento puede cambiar de una vez a otra.


In [ ]:
# Lanza programas y lee las variables del sistema.
import os
import subprocess
# Permite subir de carpeta hasta encontrar la app.
from pathlib import Path

# Busca la carpeta de la app, para que Node encuentre sus paquetes.
app = Path.cwd()
while app != app.parent and not (app / "src" / "logic").is_dir():
    app = app.parent
if not (app / "src" / "logic").is_dir():
    raise SystemExit("Cannot find the app.")

# El programa de Node que llama a la API.
code = r"""
// Pinta tablas en la salida del cuaderno.
import Table from "cli-table3";

// Dirección de la API.
const ORIGIN = "http://127.0.0.1:3004";

// Lanza la petición.
const response = await fetch(`${ORIGIN}/users/gender-count`);

// Muestra el método, la ruta y el estado de la respuesta.
console.log(`GET /users/gender-count: ${response.status}`);

// Lee la respuesta. Algunas respuestas no traen nada.
const text = await response.text();
const result = text === "" ? null : JSON.parse(text);

// Si ha fallado, se ve el mensaje y no la tabla.
if (!response.ok) {
  console.log(result?.message ?? "");
  process.exit(0);
}

// Prepara la tabla con los tres recuentos.
const table = new Table({
  head: ["women", "men", "genderUndefined"],
  style: { head: ["cyan", "bold"] },
});
table.push([result.women, result.men, result.genderUndefined]);
console.log(table.toString());
"""

# Pide a Node que escriba con colores.
env = os.environ.copy()
env["FORCE_COLOR"] = "1"
env.pop("NO_COLOR", None)

# Lanza Node y muestra lo que escribe.
completed = subprocess.run(
    ["node", "--input-type=module", "-e", code],
    cwd=app,
    capture_output=True,
    text=True,
    env=env,
)
print(completed.stdout, end="")
if completed.returncode != 0:
    print(completed.stderr, end="")
